### RAG Pipelines - Data Ingestion to Vector DB Pipeline

In [1]:
import os
from langchain_community.document_loaders import PyPDFLoader,PyMuPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from pathlib import Path

C:\Users\Rahul\AppData\Local\Temp\ipykernel_22560\4064311150.py:2: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import PyPDFLoader,PyMuPDFLoader
c:\Users\Rahul\RAG-Learning\.venv\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
## Read all pdf's present in the directory

def process_all_pdfs(pdf_directory):
    """Process all PDF files in a directory"""
    all_documents = []
    pdf_dir = Path(pdf_directory)
    
    # Find all PDF files recursively
    pdf_files = list(pdf_dir.glob("**/*.pdf"))
    
    print(f"Found {len(pdf_files)} PDF files to process")
    
    for pdf_file in pdf_files:
        print(f"\nProcessing: {pdf_file.name}")
        try:
            loader = PyPDFLoader(str(pdf_file))
            documents = loader.load()
            
            # Add source information to metadata
            for doc in documents:
                doc.metadata['source_file'] = pdf_file.name
                doc.metadata['file_type'] = 'pdf'
            
            all_documents.extend(documents)
            print(f"  ✓ Loaded {len(documents)} pages")
            
        except Exception as e:
            print(f"  ✗ Error: {e}")
    
    print(f"\nTotal documents loaded: {len(all_documents)}")
    return all_documents

# Process all PDFs in the data directory
all_pdf_documents = process_all_pdfs("../data")

Found 2 PDF files to process

Processing: sample pdf.pdf


Ignoring wrong pointing object 8 0 (offset 0)


  ✓ Loaded 5 pages

Processing: sample-local-pdf.pdf
  ✓ Loaded 3 pages

Total documents loaded: 8


In [3]:
all_pdf_documents

[Document(metadata={'producer': 'GNU Ghostscript 7.05', 'creator': 'Pdf995', 'creationdate': '12/12/2003 17:30:12', 'title': 'PDF', 'author': 'Software 995', 'subject': 'Create PDF with Pdf 995', 'keywords': 'pdf, create pdf, software, acrobat, adobe', 'source': '..\\data\\pdf\\sample pdf.pdf', 'total_pages': 5, 'page': 0, 'page_label': '1', 'source_file': 'sample pdf.pdf', 'file_type': 'pdf'}, page_content='The pdf995 suite of products - Pdf995, PdfEdit995, and Signature995 - is a complete solution for your document publishing needs. It \nprovides ease of use, flexibility in format, and industry-standard security- and all at no cost to you. \nPdf995 makes it easy and affordable to create professional-quality documents in the popular PDF file format. Its easy-to-use interface \nhelps you to create PDF files by simply selecting the "print" command from any application, creating documents which can be viewed \non any computer with a PDF viewer. Pdf995 supports network file saving, fast u

In [4]:
### Text splitting get into chunks

def split_documents(documents,chunk_size=1000,chunk_overlap=200):
    """Split documents into smaller chunks for better RAG performance"""
    text_splitter = RecursiveCharacterTextSplitter(
        chunk_size=chunk_size,
        chunk_overlap=chunk_overlap,
        length_function=len,
        separators=["\n\n", "\n", " ", ""]
    )
    split_docs = text_splitter.split_documents(documents)
    print(f"Split {len(documents)} documents into {len(split_docs)} chunks")
    
    # Show example of a chunk
    if split_docs:
        print(f"\nExample chunk:")
        print(f"Content: {split_docs[0].page_content[:200]}...")
        print(f"Metadata: {split_docs[0].metadata}")
    
    return split_docs

In [5]:
chunks = split_documents(all_pdf_documents)
chunks

Split 8 documents into 30 chunks

Example chunk:
Content: The pdf995 suite of products - Pdf995, PdfEdit995, and Signature995 - is a complete solution for your document publishing needs. It 
provides ease of use, flexibility in format, and industry-standard ...
Metadata: {'producer': 'GNU Ghostscript 7.05', 'creator': 'Pdf995', 'creationdate': '12/12/2003 17:30:12', 'title': 'PDF', 'author': 'Software 995', 'subject': 'Create PDF with Pdf 995', 'keywords': 'pdf, create pdf, software, acrobat, adobe', 'source': '..\\data\\pdf\\sample pdf.pdf', 'total_pages': 5, 'page': 0, 'page_label': '1', 'source_file': 'sample pdf.pdf', 'file_type': 'pdf'}


[Document(metadata={'producer': 'GNU Ghostscript 7.05', 'creator': 'Pdf995', 'creationdate': '12/12/2003 17:30:12', 'title': 'PDF', 'author': 'Software 995', 'subject': 'Create PDF with Pdf 995', 'keywords': 'pdf, create pdf, software, acrobat, adobe', 'source': '..\\data\\pdf\\sample pdf.pdf', 'total_pages': 5, 'page': 0, 'page_label': '1', 'source_file': 'sample pdf.pdf', 'file_type': 'pdf'}, page_content='The pdf995 suite of products - Pdf995, PdfEdit995, and Signature995 - is a complete solution for your document publishing needs. It \nprovides ease of use, flexibility in format, and industry-standard security- and all at no cost to you. \nPdf995 makes it easy and affordable to create professional-quality documents in the popular PDF file format. Its easy-to-use interface \nhelps you to create PDF files by simply selecting the "print" command from any application, creating documents which can be viewed \non any computer with a PDF viewer. Pdf995 supports network file saving, fast u

### embedding and vectorStoreDB

In [6]:
import numpy as np
from sentence_transformers import SentenceTransformer
import chromadb
from chromadb.config import Settings
import uuid
from typing import List, Dict, Any, Tuple
from sklearn.metrics.pairwise import cosine_similarity

In [7]:
class EmbeddingManager:
    """Handles document embedding generation using SentenceTransformer"""
    
    def __init__(self, model_name: str = "all-MiniLM-L6-v2"):
        """
        Initialize the embedding manager
        
        Args:
            model_name: HuggingFace model name for sentence embeddings
        """
        self.model_name = model_name
        self.model = None
        self._load_model()

    def _load_model(self):
        """Load the SentenceTransformer model"""
        try:
            print(f"Loading embedding model: {self.model_name}")
            self.model = SentenceTransformer(self.model_name)
            print(f"Model loaded successfully. Embedding dimension: {self.model.get_embedding_dimension()}")
        except Exception as e:
            print(f"Error loading model {self.model_name}: {e}")
            raise

    def generate_embeddings(self, texts: List[str]) -> np.ndarray:
        """
        Generate embeddings for a list of texts
        
        Args:
            texts: List of text strings to embed
            
        Returns:
            numpy array of embeddings with shape (len(texts), embedding_dim)
        """
        if not self.model:
            raise ValueError("Model not loaded")
        
        print(f"Generating embeddings for {len(texts)} texts...")
        embeddings = self.model.encode(texts, show_progress_bar=True)
        print(f"Generated embeddings with shape: {embeddings.shape}")
        return embeddings


## initialize the embedding manager

embedding_manager=EmbeddingManager()
embedding_manager

Loading embedding model: all-MiniLM-L6-v2


Loading weights: 100%|██████████| 103/103 [00:00<00:00, 7357.17it/s]


Model loaded successfully. Embedding dimension: 384


### VectorStore

In [8]:
class VectorStore:
    """Manages document embeddings in a ChromaDB vector store"""
    
    def __init__(self, collection_name: str = "pdf_documents", persist_directory: str = "../data/vector_store"):
        """
        Initialize the vector store
        
        Args:
            collection_name: Name of the ChromaDB collection
            persist_directory: Directory to persist the vector store
        """
        self.collection_name = collection_name
        self.persist_directory = persist_directory
        self.client = None
        self.collection = None
        self._initialize_store()

    def _initialize_store(self):
        """Initialize ChromaDB client and collection"""
        try:
            # Create persistent ChromaDB client
            os.makedirs(self.persist_directory, exist_ok=True)
            self.client = chromadb.PersistentClient(path=self.persist_directory)
            
            # Get or create collection
            self.collection = self.client.get_or_create_collection(
                name=self.collection_name,
                metadata={"description": "PDF document embeddings for RAG"}
            )
            print(f"Vector store initialized. Collection: {self.collection_name}")
            print(f"Existing documents in collection: {self.collection.count()}")
            
        except Exception as e:
            print(f"Error initializing vector store: {e}")
            raise

    def add_documents(self, documents: List[Any], embeddings: np.ndarray):
        """
        Add documents and their embeddings to the vector store
        
        Args:
            documents: List of LangChain documents
            embeddings: Corresponding embeddings for the documents
        """
        if len(documents) != len(embeddings):
            raise ValueError("Number of documents must match number of embeddings")
        
        print(f"Adding {len(documents)} documents to vector store...")
        
        # Prepare data for ChromaDB
        ids = []
        metadatas = []
        documents_text = []
        embeddings_list = []
        
        for i, (doc, embedding) in enumerate(zip(documents, embeddings)):
            # Generate unique ID
            doc_id = f"doc_{uuid.uuid4().hex[:8]}_{i}"
            ids.append(doc_id)
            
            # Prepare metadata
            metadata = dict(doc.metadata)
            metadata['doc_index'] = i
            metadata['content_length'] = len(doc.page_content)
            metadatas.append(metadata)
            
            # Document content
            documents_text.append(doc.page_content)
            
            # Embedding
            embeddings_list.append(embedding.tolist())
        
        # Add to collection
        try:
            self.collection.add(
                ids=ids,
                embeddings=embeddings_list,
                metadatas=metadatas,
                documents=documents_text
            )
            print(f"Successfully added {len(documents)} documents to vector store")
            print(f"Total documents in collection: {self.collection.count()}")
            
        except Exception as e:
            print(f"Error adding documents to vector store: {e}")
            raise

vectorstore=VectorStore()
vectorstore

Vector store initialized. Collection: pdf_documents
Existing documents in collection: 60


In [9]:
chunks

[Document(metadata={'producer': 'GNU Ghostscript 7.05', 'creator': 'Pdf995', 'creationdate': '12/12/2003 17:30:12', 'title': 'PDF', 'author': 'Software 995', 'subject': 'Create PDF with Pdf 995', 'keywords': 'pdf, create pdf, software, acrobat, adobe', 'source': '..\\data\\pdf\\sample pdf.pdf', 'total_pages': 5, 'page': 0, 'page_label': '1', 'source_file': 'sample pdf.pdf', 'file_type': 'pdf'}, page_content='The pdf995 suite of products - Pdf995, PdfEdit995, and Signature995 - is a complete solution for your document publishing needs. It \nprovides ease of use, flexibility in format, and industry-standard security- and all at no cost to you. \nPdf995 makes it easy and affordable to create professional-quality documents in the popular PDF file format. Its easy-to-use interface \nhelps you to create PDF files by simply selecting the "print" command from any application, creating documents which can be viewed \non any computer with a PDF viewer. Pdf995 supports network file saving, fast u

In [10]:
### convert text to embeddings
texts = [doc.page_content for doc in chunks]

## generate the embeddings
embeddings = embedding_manager.generate_embeddings(texts)

## store in the vector database
vectorstore.add_documents(chunks,embeddings)

Generating embeddings for 30 texts...


Batches: 100%|██████████| 1/1 [00:00<00:00,  1.10it/s]

Generated embeddings with shape: (30, 384)
Adding 30 documents to vector store...
Successfully added 30 documents to vector store
Total documents in collection: 90


Retriever Pipeline From VectorStore

In [11]:
class RAGRetriever:
    """Handles query-based retrieval from the vector store"""
    
    def __init__(self, vector_store: VectorStore, embedding_manager: EmbeddingManager):
        """
        Initialize the retriever
        
        Args:
            vector_store: Vector store containing document embeddings
            embedding_manager: Manager for generating query embeddings
        """
        self.vector_store = vector_store
        self.embedding_manager = embedding_manager

    def retrieve(self, query: str, top_k: int = 5, score_threshold: float = 0.0) -> List[Dict[str, Any]]:
        """
        Retrieve relevant documents for a query
        
        Args:
            query: The search query
            top_k: Number of top results to return
            score_threshold: Minimum similarity score threshold
            
        Returns:
            List of dictionaries containing retrieved documents and metadata
        """
        print(f"Retrieving documents for query: '{query}'")
        print(f"Top K: {top_k}, Score threshold: {score_threshold}")
        
        # Generate query embedding
        query_embedding = self.embedding_manager.generate_embeddings([query])[0]
        
        # Search in vector store
        try:
            results = self.vector_store.collection.query(
                query_embeddings=[query_embedding.tolist()],
                n_results=top_k
            )
            
            # Process results
            retrieved_docs = []
            
            if results['documents'] and results['documents'][0]:
                documents = results['documents'][0]
                metadatas = results['metadatas'][0]
                distances = results['distances'][0]
                ids = results['ids'][0]
                
                for i, (doc_id, document, metadata, distance) in enumerate(zip(ids, documents, metadatas, distances)):
                    # Convert distance to similarity score (ChromaDB uses cosine distance)
                    similarity_score = 1 - distance
                    
                    if similarity_score >= score_threshold:
                        retrieved_docs.append({
                            'id': doc_id,
                            'content': document,
                            'metadata': metadata,
                            'similarity_score': similarity_score,
                            'distance': distance,
                            'rank': i + 1
                        })
                
                print(f"Retrieved {len(retrieved_docs)} documents (after filtering)")
            else:
                print("No documents found")
            
            return retrieved_docs
            
        except Exception as e:
            print(f"Error during retrieval: {e}")
            return []

rag_retriever=RAGRetriever(vectorstore,embedding_manager)

In [12]:
rag_retriever

In [13]:
rag_retriever.retrieve("What is the author's opinion about whether pages can be longer if they are the same size?")

Retrieving documents for query: 'What is the author's opinion about whether pages can be longer if they are the same size?'
Top K: 5, Score threshold: 0.0
Generating embeddings for 1 texts...


Batches: 100%|██████████| 1/1 [00:00<00:00, 62.51it/s]

Generated embeddings with shape: (1, 384)
Retrieved 3 documents (after filtering)


[{'id': 'doc_f161e1fc_15',
  'content': 'Sample PDF  Created for testing PDFObject  This PDF is three pages long. Three long pages. Or three short pages if you’re optimistic. Is it the same as saying “three long minutes”, knowing that all minutes are the same duration, and one cannot possibly be longer than the other? If these pages are all the same size, can one possibly be longer than the other?  I digress. Here’s some Latin. Lorem ipsum dolor sit amet, consectetur adipiscing elit. Integer nec odio. Praesent libero. Sed cursus ante dapibus diam. Sed nisi. Nulla quis sem at nibh elementum imperdiet. Duis sagittis ipsum. Praesent mauris. Fusce nec tellus sed augue semper porta. Mauris massa. Vestibulum lacinia arcu eget nulla. Class aptent taciti sociosqu ad litora torquent per conubia nostra, per inceptos himenaeos. Curabitur sodales ligula in libero.   Sed dignissim lacinia nunc. Curabitur tortor. Pellentesque nibh. Aenean quam. In scelerisque sem at dolor. Maecenas mattis. Sed conva

In [14]:
rag_retriever.retrieve("VRML Mission Statement")

Retrieving documents for query: 'VRML Mission Statement'
Top K: 5, Score threshold: 0.0
Generating embeddings for 1 texts...


Batches: 100%|██████████| 1/1 [00:00<00:00, 111.09it/s]

Generated embeddings with shape: (1, 384)
Retrieved 5 documents (after filtering)


[{'id': 'doc_0b7a8ee5_10',
  'content': "Virtual Reality Markup Language (VRML) was coined, and the group resolved to begin\nspecification work after the conference. The word 'Markup'was later changed to\n'Modeling'to reflect the graphical nature of VRML.",
  'metadata': {'title': 'PDF',
   'content_length': 212,
   'page_label': '3',
   'doc_index': 10,
   'producer': 'GNU Ghostscript 7.05',
   'page': 2,
   'file_type': 'pdf',
   'creator': 'Pdf995',
   'source_file': 'sample pdf.pdf',
   'source': '..\\data\\pdf\\sample pdf.pdf',
   'total_pages': 5,
   'keywords': 'pdf, create pdf, software, acrobat, adobe',
   'author': 'Software 995',
   'creationdate': '12/12/2003 17:30:12',
   'subject': 'Create PDF with Pdf 995'},
  'similarity_score': 0.08059775829315186,
  'distance': 0.9194022417068481,
  'rank': 1},
 {'id': 'doc_2cb4c560_10',
  'content': "Virtual Reality Markup Language (VRML) was coined, and the group resolved to begin\nspecification work after the conference. The word '

### Integration VectorDB Context Pipeline With LLM Output

In [19]:
### RAG Pipeline With Groq LLM 
from langchain_groq import ChatGroq
import os
from dotenv import load_dotenv
load_dotenv()


## Initialize the GROQ LLM
groq_api_key=os.getenv("GROQ_API_KEY")

llm = ChatGroq(groq_api_key=groq_api_key,model="llama-3.3-70b-versatile",temperature=0.1,max_tokens=1024)

## Simple RAG Function: Retrieve Context + Generate Response
def rag_simple(query,retriever,llm,top_k=3):
    ## Retrieve the context
    results = retriever.retrieve(query,top_k=top_k)
    context ="\n\n".join([doc['content'] for doc in results]) if results else ""

    if not context:
        return "No Relevant Context found to answer this question"
    
    ## Generate the answer using GROQ LLM
    prompt = f"""use the following context to answer the question concisely.
    context: {context}
    question:{query}
"""
    
    response = llm.invoke([prompt.format(context=context,query=query)])
    return response.content

In [20]:
answer=rag_simple("What is VRML Mission Statement",rag_retriever,llm)

Retrieving documents for query: 'What is VRML Mission Statement'
Top K: 3, Score threshold: 0.0
Generating embeddings for 1 texts...


Batches: 100%|██████████| 1/1 [00:00<00:00, 64.46it/s]

Generated embeddings with shape: (1, 384)
Retrieved 3 documents (after filtering)


In [21]:
answer

"The context doesn't explicitly state the VRML mission statement. It only mentions the coinage of VRML, the start of specification work, and the change from 'Markup' to 'Modeling'."

### Enhanced RAG Pipeline Features

In [23]:
# --- Enhanced RAG Pipeline Features ---
def rag_advanced(query, retriever, llm, top_k=5, min_score=0.2, return_context=False):
    """
    RAG pipeline with extra features:
    - Returns answer, sources, confidence score, and optionally full context.
    """
    results = retriever.retrieve(query, top_k=top_k, score_threshold=min_score)
    if not results:
        return {'answer': 'No relevant context found.', 'sources': [], 'confidence': 0.0, 'context': ''}
    
    # Prepare context and sources
    context = "\n\n".join([doc['content'] for doc in results])
    sources = [{
        'source': doc['metadata'].get('source_file', doc['metadata'].get('source', 'unknown')),
        'page': doc['metadata'].get('page', 'unknown'),
        'score': doc['similarity_score'],
        'preview': doc['content'][:300] + '...'
    } for doc in results]
    confidence = max([doc['similarity_score'] for doc in results])
    
    # Generate answer
    prompt = f"""Use the following context to answer the question concisely.\nContext:\n{context}\n\nQuestion: {query}\n\nAnswer:"""
    response = llm.invoke([prompt.format(context=context, query=query)])
    
    output = {
        'answer': response.content,
        'sources': sources,
        'confidence': confidence
    }
    if return_context:
        output['context'] = context
    return output

# Example usage:
result = rag_advanced("What is VRML Mission Statement", rag_retriever, llm, top_k=3, min_score=0.1, return_context=True)
print("Answer:", result['answer'])
print("Sources:", result['sources'])
print("Confidence:", result['confidence'])
print("Context Preview:", result['context'][:300])

Retrieving documents for query: 'What is VRML Mission Statement'
Top K: 3, Score threshold: 0.1
Generating embeddings for 1 texts...


Batches: 100%|██████████| 1/1 [00:00<00:00, 71.44it/s]

Generated embeddings with shape: (1, 384)
Retrieved 3 documents (after filtering)


Answer: There is no VRML mission statement provided in the given context. The context only discusses the coining of the term "Virtual Reality Markup Language (VRML)" and the later change to "Virtual Reality Modeling Language" to reflect its graphical nature.
Sources: [{'source': 'sample pdf.pdf', 'page': 2, 'score': 0.19900768995285034, 'preview': "Virtual Reality Markup Language (VRML) was coined, and the group resolved to begin\nspecification work after the conference. The word 'Markup'was later changed to\n'Modeling'to reflect the graphical nature of VRML...."}, {'source': 'sample pdf.pdf', 'page': 2, 'score': 0.19900768995285034, 'preview': "Virtual Reality Markup Language (VRML) was coined, and the group resolved to begin\nspecification work after the conference. The word 'Markup'was later changed to\n'Modeling'to reflect the graphical nature of VRML...."}, {'source': 'sample pdf.pdf', 'page': 2, 'score': 0.19900768995285034, 'preview': "Virtual Reality Markup Language (VRML) was 

In [24]:
# --- Advanced RAG Pipeline: Streaming, Citations, History, Summarization ---
from typing import List, Dict, Any
import time

class AdvancedRAGPipeline:
    def __init__(self, retriever, llm):
        self.retriever = retriever
        self.llm = llm
        self.history = []  # Store query history

    def query(self, question: str, top_k: int = 5, min_score: float = 0.2, stream: bool = False, summarize: bool = False) -> Dict[str, Any]:
        # Retrieve relevant documents
        results = self.retriever.retrieve(question, top_k=top_k, score_threshold=min_score)
        if not results:
            answer = "No relevant context found."
            sources = []
            context = ""
        else:
            context = "\n\n".join([doc['content'] for doc in results])
            sources = [{
                'source': doc['metadata'].get('source_file', doc['metadata'].get('source', 'unknown')),
                'page': doc['metadata'].get('page', 'unknown'),
                'score': doc['similarity_score'],
                'preview': doc['content'][:120] + '...'
            } for doc in results]
            # Streaming answer simulation
            prompt = f"""Use the following context to answer the question concisely.\nContext:\n{context}\n\nQuestion: {question}\n\nAnswer:"""
            if stream:
                print("Streaming answer:")
                for i in range(0, len(prompt), 80):
                    print(prompt[i:i+80], end='', flush=True)
                    time.sleep(0.05)
                print()
            response = self.llm.invoke([prompt.format(context=context, question=question)])
            answer = response.content

        # Add citations to answer
        citations = [f"[{i+1}] {src['source']} (page {src['page']})" for i, src in enumerate(sources)]
        answer_with_citations = answer + "\n\nCitations:\n" + "\n".join(citations) if citations else answer

        # Optionally summarize answer
        summary = None
        if summarize and answer:
            summary_prompt = f"Summarize the following answer in 2 sentences:\n{answer}"
            summary_resp = self.llm.invoke([summary_prompt])
            summary = summary_resp.content

        # Store query history
        self.history.append({
            'question': question,
            'answer': answer,
            'sources': sources,
            'summary': summary
        })

        return {
            'question': question,
            'answer': answer_with_citations,
            'sources': sources,
            'summary': summary,
            'history': self.history
        }

# Example usage:
adv_rag = AdvancedRAGPipeline(rag_retriever, llm)
result = adv_rag.query("what is VRML", top_k=3, min_score=0.1, stream=True, summarize=True)
print("\nFinal Answer:", result['answer'])
print("Summary:", result['summary'])
print("History:", result['history'][-1])

Retrieving documents for query: 'what is VRML'
Top K: 3, Score threshold: 0.1
Generating embeddings for 1 texts...


Batches: 100%|██████████| 1/1 [00:00<00:00, 76.93it/s]

Generated embeddings with shape: (1, 384)
Retrieved 3 documents (after filtering)
Streaming answer:
Use the following context to answer the question concisely.
Context:
APPROVED
Introduction
The Virtual Reality Modeling Language (VRML) is a language for describing multi-
participant interactive simulations -- virtual worlds networked via the global Internet and
hyperlinked with the World Wide Web. All aspects of virt

ual world display, interaction
and internetworking can be specified using VRML. It is the intention of its designers that
VRML become the standard language for interactive simulation within the World Wide
Web.
The first version of VRML allows for the creation of virtual worlds with limited
interactive behavior. These worlds can contain objects which have hyperlinks to other
worlds, HTML documents or other valid MIME types. When the user selects an object
with a hyperlink, the appropriate MIME viewer is launched. When the user selects a link
to a VRML document from within a correctly configured WWW browser, a VRML
viewer is launched. Thus VRML viewers are the perfect companion applications to

APPROVED
Introduction
The Virtual Reality Modeling Language (VRML) is a language for describing multi-
participant interactive simulations -- virtual worlds networked via the global Internet and
hyperlinked with the World Wide Web. All aspects of virtual world display, interaction
and internetwork